In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,expr,col,when,concat,hash,concat_ws, array, lpad,size,array_intersect,regexp_replace
from pyspark.sql.types import ShortType, ArrayType, IntegerType

In [4]:

spark = SparkSession.builder.appName("etl").config("spark.driver.memory",'4g').getOrCreate()


In [5]:
spark

In [6]:
df_pyspark = spark.read.option('multiline','True').json('/home/milan-thapa/Desktop/Zaki_point_task/files/output/data.json')
df1_pyspark = spark.read.parquet('/home/milan-thapa/Desktop/Zaki_point_task/files/highmark_prv')

25/05/27 13:03:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
df_pyspark.printSchema()
df1_pyspark.printSchema()


root
 |-- in_network: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- billing_code: string (nullable = true)
 |    |    |-- billing_code_type: string (nullable = true)
 |    |    |-- billing_code_type_version: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- negotiated_rates: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |    |    |-- expiration_date: strin

In [8]:
np_data = (
    df_pyspark.selectExpr("*", "explode(in_network) as net").drop("in_network")
    .select("*", "net.*").drop("net")
    .selectExpr("*", "explode(negotiated_rates) as rates").drop("negotiated_rates")
    .selectExpr("*", "explode(rates.provider_groups) as id").drop("provider_groups")
    .selectExpr("*", "explode(id.npi) as npi", "id.tin.type as tin_type", "id.tin.value as tin").drop("id")
    .selectExpr("*", "explode(rates.negotiated_prices) as prices").drop("rates")
    .select("*", "prices.*").drop("prices")
)
np_data.printSchema()


root
 |-- last_updated_on: string (nullable = true)
 |-- reporting_entity_name: string (nullable = true)
 |-- reporting_entity_type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)
 |-- additional_information: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- expiration_date: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (c

In [9]:
provider_cleaned = np_data.withColumn('tin', expr("REPLACE(tin, '-', '')"))


In [10]:
provider_replaced = np_data.withColumn("tin_type",
                    when(col("tin_type") == "ein", 1)
                    .when(col("tin_type") == "npi", 2))


In [46]:
df_hash = provider_replaced.withColumn('provider_group_id',hash(concat("npi", "tin")))
    
in_net = df_hash.select(
    "billing_code",
    "billing_code_type",
    "negotiation_arrangement",
    "provider_group_id",
    "billing_class",
    "billing_code_modifier",
    "negotiated_rate",
    "negotiated_type",
    "service_code"
    )
in_net.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- provider_group_id: integer (nullable = false)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [ ]:
df_combine = np_data.withColumn("provider_group_id",concat("npi", "tin"))


In [ ]:
df_hash = df_combine.withColumn('provider_group_id',hash("provider_group_id"))


In [13]:
df_nr_neww = df_combine.drop('npi', 'tin_type', 'tin')

In [14]:
in_net = df_nr_neww.select(
    "billing_code",
    "billing_code_type",
    "negotiation_arrangement",
    "provider_group_id",
    "billing_class",
    "billing_code_modifier",
    "negotiated_rate",
    "negotiated_type",
    "service_code"
    )

np_data = (in_net.filter(in_net.billing_code.isNotNull() & (in_net.billing_code != ""))
            .withColumn("service_code",col("service_code").cast(ArrayType(IntegerType())))
)

In [15]:
# df_nr_neww.show()

In [16]:
df_nr_neww.printSchema()  
# Nr of mrf file

root
 |-- last_updated_on: string (nullable = true)
 |-- reporting_entity_name: string (nullable = true)
 |-- reporting_entity_type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- additional_information: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- expiration_date: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- provider_group_id: string (nullable = true)



In [17]:
provider_table = df_hash.selectExpr( 'npi',
            "tin_type",
            "tin",
            "provider_group_id")

In [18]:
provider_table.printSchema()

root
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)
 |-- provider_group_id: integer (nullable = false)



In [19]:
remove_ = provider_table.withColumn('tin', expr("replace(tin,'-','')"))
df_type = remove_.withColumn('tin_type',
        when((col('tin_type')== 'ein'), 1).when((col('tin_type')== 'npi'), 2)
)

In [20]:
provider_rep = df_type.withColumn("tin_type", col("tin_type").cast(ShortType()))

In [21]:
provider = provider_rep.select(
    "provider_group_id",
    "npi",
    "tin_type",
    "tin"
)

In [22]:
# provider_rep.show()
# pr of mrf file 

In [23]:
df_dropping = df1_pyspark.drop('prv_fax','provider_name_prefix_text','prv_type_desc')
df_new = df_dropping.withColumn('prv_type_code',
                             when((col('prv_type_code')== 'P'), 1).when((col('prv_type_code')== 'F'), 2)
)

In [24]:
df_cast = df_new.withColumn("prv_type_code",col("prv_type_code").cast(IntegerType()))
df_merge = df_cast.withColumn("full_name",concat_ws(" ","provider_first_name", "provider_middle_name","provider_last_name"))
df_merge1=df_merge.drop("provider_first_name","provider_last_name","provider_middle_name")


In [25]:
df_new1 = df_merge1.select(
    "*",  
    col("loc.lat").alias("latitude"),
    col("loc.lon").alias("longitude")
)

In [26]:
df_new2 = df_new1.drop('loc')

df_array = df_new2.withColumn("taxonomy",array(col("prv_taxonomy_1_code"),col("prv_taxonomy_2_code"),col("prv_taxonomy_3_code")))
df_array1=df_array.drop("prv_taxonomy_1_code","prv_taxonomy_2_code","prv_taxonomy_3_code")


In [27]:
df_array2 = df_array1.withColumn("prv_specialty",array(col("prv_specialty_1_desc"),col("prv_specialty_2_desc"),col("prv_specialty_3_desc")))
df_array3=df_array2.drop("prv_specialty_1_desc","prv_specialty_2_desc","prv_specialty_3_desc")



In [28]:
# df_array3.show()
# df_array3.printSchema()
#new pr table

In [29]:
df_join = provider_rep.join(df_array3,how="inner",on=["npi","tin"])
# joining pr table

In [30]:
df_join.printSchema()

root
 |-- npi: long (nullable = true)
 |-- tin: string (nullable = true)
 |-- tin_type: short (nullable = true)
 |-- provider_group_id: integer (nullable = false)
 |-- prv_street_1: string (nullable = true)
 |-- prv_city: string (nullable = true)
 |-- prv_state: string (nullable = true)
 |-- prv_zip: string (nullable = true)
 |-- prv_phone: string (nullable = true)
 |-- prv_type_code: integer (nullable = true)
 |-- full_name: string (nullable = false)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- taxonomy: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- prv_specialty: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [31]:
df_bil_code = spark.read.option('header','True').csv('/home/milan-thapa/Desktop/Zaki_point_task/files/billing_taxonomy_list (2).csv')
df_bil_code.printSchema()
# reading csv file

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_description: string (nullable = true)
 |-- taxonomy_list: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)



In [32]:
df_leftpad = df_bil_code.withColumn("billing_code", lpad(col("billing_code"), 5, "0"))
df_rate2 = df_leftpad.drop('_c4','_c5','_c6') \
    .withColumn("taxonomy_list", array(regexp_replace(col("taxonomy_list"), r"^\{|\}$", ""))) 
df_join = df_rate2.select(
    "billing_code",
    "taxonomy_list"
    )

In [33]:
df_rate2.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_description: string (nullable = true)
 |-- taxonomy_list: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [34]:
df_join.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- taxonomy_list: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [35]:
df_leftpad = df_bil_code.withColumn("billing_code", lpad(col("billing_code"), 5, "0")) \
.drop('_c4','_c5','_c6') \
.withColumn("taxonomy_list", array(regexp_replace(col("taxonomy_list"), r"^\{|\}$", "")))

In [36]:
df_rate2.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_description: string (nullable = true)
 |-- taxonomy_list: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [37]:
df_join = df_rate2.select(
    "billing_code",
    "taxonomy_list"
    )

In [38]:
df_join.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- taxonomy_list: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [39]:
df_newrtab = df_nr_neww.join(df_join,on="billing_code",how="inner")

In [40]:
df_newrtab.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- last_updated_on: string (nullable = true)
 |-- reporting_entity_name: string (nullable = true)
 |-- reporting_entity_type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- additional_information: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- expiration_date: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- provider_group_id: string (nullable = true)
 |-- taxonomy_list: array (nullable = f

In [41]:
df_pr_join = provider_rep.join(df_array3,how="inner",on=["npi","tin"])

In [45]:
df_pr_join.show(2)

+----------+----------+--------+-----------------+-------------------+-----------+---------+-------+---------+-------------+----------------+---------+----------+--------------------+--------------------+
|       npi|       tin|tin_type|provider_group_id|       prv_street_1|   prv_city|prv_state|prv_zip|prv_phone|prv_type_code|       full_name| latitude| longitude|            taxonomy|       prv_specialty|
+----------+----------+--------+-----------------+-------------------+-----------+---------+-------+---------+-------------+----------------+---------+----------+--------------------+--------------------+
|1003006735|1134494909|       2|       1509708653|1001 BALTIMORE PIKE|SPRINGFIELD|       PA|  19064|     NULL|            1|Nicole M. KOEPKE|  39.9144| -75.34188|[363LP2300X, NULL, ]|[Primary Care Nur...|
|1003006735|1134494909|       2|       1509708653| 1437 DEKALB STREET| NORRISTOWN|       PA|  19401|     NULL|            1|Nicole M. KOEPKE|40.124271|-75.333008|[363LP2300X, NULL,

In [42]:
nrpr = df_newrtab.join(df_pr_join,on="provider_group_id",how="inner")

In [44]:
nrpr.show(2)

+-----------------+------------+---------------+---------------------+---------------------+-------+-----------------+-------------------------+-----------+----+-----------------------+----------------------+-------------+---------------------+---------------+---------------+---------------+------------+-------------+---+---+--------+------------+--------+---------+-------+---------+-------------+---------+--------+---------+--------+-------------+
|provider_group_id|billing_code|last_updated_on|reporting_entity_name|reporting_entity_type|version|billing_code_type|billing_code_type_version|description|name|negotiation_arrangement|additional_information|billing_class|billing_code_modifier|expiration_date|negotiated_rate|negotiated_type|service_code|taxonomy_list|npi|tin|tin_type|prv_street_1|prv_city|prv_state|prv_zip|prv_phone|prv_type_code|full_name|latitude|longitude|taxonomy|prv_specialty|
+-----------------+------------+---------------+---------------------+---------------------+--

In [43]:
specialized_filter = nrpr.filter(size(array_intersect(col("taxonomy"), col("taxonomy_list"))) > 0)
specialized_filter.show(2)

25/05/27 13:03:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+------------+---------------+---------------------+---------------------+-------+-----------------+-------------------------+-----------+----+-----------------------+----------------------+-------------+---------------------+---------------+---------------+---------------+------------+-------------+---+---+--------+------------+--------+---------+-------+---------+-------------+---------+--------+---------+--------+-------------+
|provider_group_id|billing_code|last_updated_on|reporting_entity_name|reporting_entity_type|version|billing_code_type|billing_code_type_version|description|name|negotiation_arrangement|additional_information|billing_class|billing_code_modifier|expiration_date|negotiated_rate|negotiated_type|service_code|taxonomy_list|npi|tin|tin_type|prv_street_1|prv_city|prv_state|prv_zip|prv_phone|prv_type_code|full_name|latitude|longitude|taxonomy|prv_specialty|
+-----------------+------------+---------------+---------------------+---------------------+--